In [104]:
import sys
sys.path.append("../worker")
import joblib
import pandas as pd
from database import engine
from sqlalchemy import text
from datetime import timedelta

In [105]:
GET_AGG_SQL = """ 
SELECT DISTINCT ON (device_id) *
    FROM dht22_aggregate
    WHERE date < CURRENT_DATE
    ORDER BY device_id, date DESC
"""

ISERT_PRED_SQL = """ 
INSERT INTO prediction (device_id, based_on_date, target_date, temp_c_max, temp_c_min, model)
    VALUES (:device_id, :based_on_date, :target_date, :temp_c_max, :temp_c_min, :model)
    ON CONFLICT (device_id, based_on_date, target_date)
    DO UPDATE SET temp_c_max = EXCLUDED.temp_c_max, temp_c_min = EXCLUDED.temp_c_min, model = EXCLUDED.model
"""

prediction_model = "ridge_v1"

In [106]:
model = joblib.load("../models/ridge_v1.joblib")
model

{'max': [Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1)],
 'predictors_max': ['temp_c_max',
  'temp_c_min',
  'temp_c_avg',
  'humidity_max',
  'humidity_min',
  'humidity_avg'],
 'min': [Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1),
  Ridge(alpha=0.1)],
 'predictors_min': ['temp_c_max',
  'temp_c_min',
  'temp_c_avg',
  'humidity_max',
  'humidity_min',
  'humidity_avg']}

In [107]:
latest_agg = pd.read_sql(text(GET_AGG_SQL), engine)

latest_agg

,device_id,date,temp_c_max,temp_c_min,temp_c_avg,humidity_max,humidity_min,humidity_avg,reading_count
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,23.1,23.1,23.1,58.0,58.0,58.0,6
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,-80.0,-80.0,-80.0,100.0,100.0,100.0,1


In [108]:
predictors_max = latest_agg[model["predictors_max"]]
predictors_min = latest_agg[model["predictors_min"]]
predictors_max

,temp_c_max,temp_c_min,temp_c_avg,humidity_max,humidity_min,humidity_avg
0,23.1,23.1,23.1,58.0,58.0,58.0
1,-80.0,-80.0,-80.0,100.0,100.0,100.0


In [109]:
day1_max_prediction = model["max"][0].predict(predictors_max)
day1_min_prediction = model["min"][0].predict(predictors_min)
day1_max_prediction

array([ 24.6023976 , -76.78648129])

In [110]:
day1_df = pd.DataFrame({
    "device_id":latest_agg["device_id"],  
    "based_on_date":latest_agg["date"],
    "target_date":latest_agg["date"] + timedelta(days=1),
    "temp_c_max": day1_max_prediction,
    "temp_c_min": day1_min_prediction,
    "model":prediction_model,
})
day1_df

,device_id,based_on_date,target_date,temp_c_max,temp_c_min,model
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-28,24.602398,19.672743,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-20,-76.786481,-79.221454,ridge_v1


In [111]:
daily_predictions = []
for i in range(7):           
    max_prediction = model["max"][i].predict(predictors_max)
    min_prediction = model["min"][i].predict(predictors_min)
    daily_predictions.append(pd.DataFrame({
    "device_id":latest_agg["device_id"],  
    "based_on_date":latest_agg["date"],
    "target_date":latest_agg["date"] + timedelta(days=i+1),
    "temp_c_max": max_prediction,
    "temp_c_min": min_prediction,
    "model":prediction_model,
    }))
daily_predictions[6]

,device_id,based_on_date,target_date,temp_c_max,temp_c_min,model
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-09-03,27.887556,19.141450,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-26,-62.431513,-66.057045,ridge_v1


In [112]:
forecast = pd.concat(daily_predictions)
forecast

,device_id,based_on_date,target_date,temp_c_max,temp_c_min,model
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-28,24.602398,19.672743,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-20,-76.786481,-79.221454,ridge_v1
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-29,25.935875,18.642405,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-21,-72.281197,-76.402074,ridge_v1
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-30,26.823155,18.652945,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-22,-68.898873,-73.276175,ridge_v1
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-31,27.262316,18.812369,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-23,-66.494658,-70.764712,ridge_v1
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-09-01,27.589657,18.977385,ridge_v1
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-24,-64.725934,-68.843315,ridge_v1


In [113]:
forecast = forecast.to_dict("records")

In [114]:
with engine.begin() as conn:
    conn.execute(text(ISERT_PRED_SQL), forecast)

In [115]:
pd.read_sql(text("SELECT * FROM prediction"), engine)

,device_id,based_on_date,target_date,temp_c_max,temp_c_min,model,created_at
0,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-28,24.60,19.67,ridge_v1,2026-09-25 18:43:01.776519+00:00
1,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-20,-76.79,-79.22,ridge_v1,2026-09-25 18:43:01.776519+00:00
2,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-29,25.94,18.64,ridge_v1,2026-09-25 18:43:01.776519+00:00
3,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-21,-72.28,-76.40,ridge_v1,2026-09-25 18:43:01.776519+00:00
4,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-30,26.82,18.65,ridge_v1,2026-09-25 18:43:01.776519+00:00
5,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-22,-68.90,-73.28,ridge_v1,2026-09-25 18:43:01.776519+00:00
6,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-08-31,27.26,18.81,ridge_v1,2026-09-25 18:43:01.776519+00:00
7,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-23,-66.49,-70.76,ridge_v1,2026-09-25 18:43:01.776519+00:00
8,43fbcfd7-7812-4b85-a885-a3beb97ca194,2026-08-27,2026-09-01,27.59,18.98,ridge_v1,2026-09-25 18:43:01.776519+00:00
9,c43b9366-5f79-4575-a8d0-222c0fe45657,2026-09-19,2026-09-24,-64.73,-68.84,ridge_v1,2026-09-25 18:43:01.776519+00:00
